# Notebook 1 — Preprocesamiento de Datos
**Proyecto:** Predicción de Incumplimiento en Créditos de Consumo

**Autor:** Elver Zúñiga

**Dataset:** Muestra representativa del sistema financiero peruano (2015-2021)

In [15]:
import sys
print(sys.executable)

C:\Users\Elver\Documents\Proyectos\202604-mle-dsrp\credit-risk-ml\.venv\Scripts\python.exe


In [16]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Configuración
RANDOM_STATE = 42
TARGET       = 'incumplimiento'
FEATURES = [
    'n_atr_1d_6', 'cl_9_1.0', 'cl_12_1.0', 'cl_18_2.0', 'cl_9_3.0',
    'ind_vjrc_36_1', 'ind_per_x9_1.0', 'prc_u_tc_cns', 'tas_u_tc',
    'c_tc_mn_24', 'c_tc_mn_24_sld', 'c_pcns', 'vr_s_t_36', 'mx_ltc_36', 'ipc_t6'
]

df = pd.read_csv('../data/df_sample.csv')
print(f'Dataset cargado: {df.shape[0]:,} filas x {df.shape[1]} columnas')
df[FEATURES + [TARGET]].head()

Dataset cargado: 41,976 filas x 29 columnas


,n_atr_1d_6,cl_9_1.0,cl_12_1.0,cl_18_2.0,cl_9_3.0,ind_vjrc_36_1,ind_per_x9_1.0,prc_u_tc_cns,tas_u_tc,c_tc_mn_24,c_tc_mn_24_sld,c_pcns,vr_s_t_36,mx_ltc_36,ipc_t6,incumplimiento
0,0,0,0,0,0,0,0,0.099200,-1.000000,0,0,2,0.000000,0.00,0.043979,0
1,0,0,0,0,0,0,0,1.000000,1.000000,0,0,2,0.000000,10000.00,0.021925,0
2,0,0,0,0,0,0,0,0.068488,0.068488,0,0,0,-0.989042,800.00,0.013649,0
3,0,0,0,0,0,0,0,0.000000,0.000000,0,0,1,-0.754436,574.53,0.021925,1
4,0,0,1,0,0,0,0,0.000000,0.000000,0,0,1,-0.680519,1725.00,0.032349,0


In [17]:
df.shape

(41976, 29)

## 1. Calidad de datos

In [18]:
# Resumen de calidad
quality = pd.DataFrame({
    'dtype'     : df[FEATURES + [TARGET]].dtypes,
    'nulos'     : df[FEATURES + [TARGET]].isnull().sum(),
    'pct_nulos' : (df[FEATURES + [TARGET]].isnull().mean() * 100).round(2),
    'unicos'    : df[FEATURES + [TARGET]].nunique(),
    'min'       : df[FEATURES + [TARGET]].min().round(4),
    'max'       : df[FEATURES + [TARGET]].max().round(4),
})
print('=== Calidad del dataset ===')
display(quality)
print(f'\n✅ Total nulos: {df[FEATURES + [TARGET]].isnull().sum().sum()}')

=== Calidad del dataset ===


,dtype,nulos,pct_nulos,unicos,min,max
n_atr_1d_6,int64,0,0.0,7,0.0000,6.000
cl_9_1.0,int64,0,0.0,2,0.0000,1.000
cl_12_1.0,int64,0,0.0,2,0.0000,1.000
cl_18_2.0,int64,0,0.0,2,0.0000,1.000
cl_9_3.0,int64,0,0.0,2,0.0000,1.000
ind_vjrc_36_1,int64,0,0.0,2,0.0000,1.000
ind_per_x9_1.0,int64,0,0.0,2,0.0000,1.000
prc_u_tc_cns,float64,0,0.0,21595,0.0000,1.000
tas_u_tc,float64,0,0.0,21595,-1.0000,1.000
c_tc_mn_24,int64,0,0.0,9,0.0000,8.000



✅ Total nulos: 0


## 2. Análisis del target — balance de clases

In [19]:
vc = df[TARGET].value_counts()
tasa = df[TARGET].mean()

vc = df[TARGET].value_counts()
tasa = df[TARGET].mean()

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Distribución del target', 'Tasa de incumplimiento'),
    specs=[[{"type": "xy"}, {"type": "domain"}]]
)

fig.add_trace(go.Bar(
    x=['Cumple (0)', 'Incumple (1)'],
    y=vc.values,
    marker_color=['#1D9E75', '#D85A30'],
    text=vc.values, textposition='outside'
), row=1, col=1)

fig.add_trace(go.Pie(
    labels=['Cumple', 'Incumple'],
    values=vc.values,
    marker_colors=['#1D9E75', '#D85A30'],
    hole=0.5,
    textinfo='label+percent'
), row=1, col=2)

fig.update_layout(
    title='Balance de clases — Incumplimiento crediticio',
    height=400, template='plotly_white', showlegend=False
)
fig.show()

print(f'Cumple   : {vc[0]:,} ({vc[0]/len(df):.1%})')
print(f'Incumple : {vc[1]:,} ({vc[1]/len(df):.1%})')
print(f'\n⚠️  Desbalance moderado — se aplicará class_weight="balanced"')

Cumple   : 32,307 (77.0%)
Incumple : 9,669 (23.0%)

⚠️  Desbalance moderado — se aplicará class_weight="balanced"


## 3. Distribución de variables numéricas

In [20]:
num_features = ['n_atr_1d_6', 'prc_u_tc_cns', 'tas_u_tc', 'c_pcns', 'vr_s_t_36', 'mx_ltc_36', 'ipc_t6']

fig = make_subplots(rows=2, cols=4,
    subplot_titles=num_features)

positions = [(1,1),(1,2),(1,3),(1,4),(2,1),(2,2),(2,3)]
colors = ['#1D9E75', '#D85A30']

for i, (feat, pos) in enumerate(zip(num_features, positions)):
    for label, color in zip([0, 1], colors):
        fig.add_trace(go.Histogram(
            x=df[df[TARGET]==label][feat],
            name=f'{"Cumple" if label==0 else "Incumple"}',
            marker_color=color, opacity=0.6,
            showlegend=(i==0)
        ), row=pos[0], col=pos[1])

fig.update_layout(
    title='Distribución de features por clase<br><sup>Verde=Cumple | Rojo=Incumple</sup>',
    height=500, template='plotly_white', barmode='overlay'
)
fig.show()

## 4. Correlación con el target

In [21]:
corrs = []
for col in FEATURES:
    r, p = stats.pointbiserialr(df[TARGET], df[col])
    corrs.append({'feature': col, 'correlacion': r, 'abs_corr': abs(r), 'significativa': p < 0.05})

corr_df = pd.DataFrame(corrs).sort_values('correlacion', ascending=True)

fig = go.Figure(go.Bar(
    x=corr_df['correlacion'],
    y=corr_df['feature'],
    orientation='h',
    marker_color=['#D85A30' if x > 0 else '#1D9E75' for x in corr_df['correlacion']],
    text=corr_df['correlacion'].round(3),
    textposition='outside'
))
fig.update_layout(
    title='Correlación point-biserial con incumplimiento<br><sup>Rojo=aumenta incumplimiento | Verde=reduce incumplimiento</sup>',
    xaxis_title='Correlación', height=500, template='plotly_white'
)
fig.show()

print('\nTop 5 features más predictivas:')
display(corr_df.sort_values('abs_corr', ascending=False).head(5)[['feature','correlacion','significativa']])


Top 5 features más predictivas:


,feature,correlacion,significativa
0,n_atr_1d_6,0.153415,True
7,prc_u_tc_cns,0.143677,True
1,cl_9_1.0,0.140390,True
2,cl_12_1.0,0.128519,True
12,vr_s_t_36,0.116934,True


## 5. Matriz de correlación entre features

In [22]:
corr_matrix = df[FEATURES].corr().round(2)

fig = go.Figure(go.Heatmap(
    z=corr_matrix.values,
    x=corr_matrix.columns,
    y=corr_matrix.columns,
    colorscale='RdBu', zmid=0, zmin=-1, zmax=1,
    text=corr_matrix.values,
    texttemplate='%{text}', textfont_size=9
))
fig.update_layout(
    title='Matriz de correlación entre features<br><sup>Valores >0.8 indican multicolinealidad</sup>',
    height=550, template='plotly_white'
)
fig.show()

# Detectar multicolinealidad
high_corr = []
for i in range(len(FEATURES)):
    for j in range(i+1, len(FEATURES)):
        c = abs(corr_matrix.iloc[i,j])
        if c > 0.7:
            high_corr.append((FEATURES[i], FEATURES[j], round(c,3)))

if high_corr:
    print('⚠️  Pares con alta correlación (>0.7):')
    for a, b, c in high_corr:
        print(f'   {a} — {b}: {c}')
else:
    print('✅ No se detectó multicolinealidad severa')

⚠️  Pares con alta correlación (>0.7):
   cl_9_1.0 — cl_12_1.0: 0.86


## 6. Exportar dataset procesado

In [23]:
from sklearn.model_selection import train_test_split

df_model = df[FEATURES + [TARGET]].copy()

X = df_model[FEATURES]
y = df_model[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Guardar splits
X_train.to_csv('../data/X_train.csv', index=False)
X_test.to_csv('../data/X_test.csv',  index=False)
y_train.to_csv('../data/y_train.csv', index=False)
y_test.to_csv('../data/y_test.csv',  index=False)

print(f'✅ Train: {X_train.shape[0]:,} filas')
print(f'✅ Test:  {X_test.shape[0]:,} filas')
print(f'\nTasa incumplimiento train: {y_train.mean():.2%}')
print(f'Tasa incumplimiento test:  {y_test.mean():.2%}')
print('\n✅ Splits exportados a ../data/')

✅ Train: 33,580 filas
✅ Test:  8,396 filas

Tasa incumplimiento train: 23.03%
Tasa incumplimiento test:  23.03%

✅ Splits exportados a ../data/
